<a href="https://colab.research.google.com/github/abubakirovx/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
print("Menu Detector")

Menu Detector


In [51]:
# IMPORT LIBRARIES
from google.colab import drive

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v2
import torchvision.transforms as transforms
from torch.utils.data import Dataset,DataLoader

from PIL import Image, UnidentifiedImageError
import os
import numpy as np



In [52]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [53]:
#Define Dataset Path
DATASET_PATH = "/content/drive/MyDrive/food101"
print("Dataset Path",DATASET_PATH)

Dataset Path /content/drive/MyDrive/food101


In [54]:
CUSTOM_CLASS_MAPPING={
    "pizza":"pizza",
    "hot_dog":"hot_dog",
    "osh":"pilaf",
    "kabob":"kebab",
    "manti":"manti",
    "chocolate_cake":"desert", # label grouping |class consolidation
    "cheesecake":"desert"      # label grouping |class consolidation
}

CLASSES =['pizza','hot_dog','pilaf','kebab','desert','manti']
CLASS_TO_IDX={cls:i for i,cls in enumerate(CLASSES)}
NUM_CLASSES=len(CLASSES)

print(CLASS_TO_IDX)
print(NUM_CLASSES)

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# 0 ~ 255 ToTensor => 0.0 ~ 0.1 | 0.234 | H, W, C => C, H, W
# Normalize => pixel = (pixel-mean) / std


{'pizza': 0, 'hot_dog': 1, 'pilaf': 2, 'kebab': 3, 'desert': 4, 'manti': 5}
6


In [55]:
# Custom Dataset Class

class FoodDataset(Dataset):
  def __init__(self,images,labels,transform=None):
    self.images=images
    self.labels=labels
    self.transform=transform

  def __len__ (self):
    return len(self.images)

  def __getitem__(self,idx):
    image_path = self.images[idx]
    label=self.labels[idx]
    try:
       image = Image.open(image_path)

       if image.mode == "P" or image.mode == "RGBA":
            image = image.convert("RGBA").convert("RGB")
       else:
         image = image.convert("RGB")
    except(UnidentifiedImageError,OSError):
        print(f"Skipping broken image: {image_path}")
        return self.__getitem__((idx+1)%len(self.images))
    if self.transform:
         image = self.transform(image)
    return image ,label

In [56]:
# Gather and Split Data

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path=os.path.join(DATASET_PATH,original_class) #/content/drive/MyDrive/food101/pizza
  if not os.path.exists(class_path):
      print(f"Warning: {class_path} is not found!")
      continue
  for img in os.listdir(class_path):
    if img.endswith(('.jpg','.jpeg','.png')):
      full_path=os.path.join(class_path,img) # /content/drive/MyDrive/food101/pizza/3102271.jpg
      all_images.append((full_path,CLASS_TO_IDX[mapped_class])) # ('/content/drive/MyDrive/food101/pizza/3102271.jpg', 0)
np.random.shuffle(all_images)
split=int(0.8*len(all_images))
train_data=all_images[:split]
val_data=all_images[split:]



tran_images,train_labels = zip(*train_data)
val_images,val_labels=zip(*val_data)

# print("all_images:",all_images)

dataset = FoodDataset(tran_images,train_labels)
print(len(dataset))
img,lbl=dataset[0]

3440


In [57]:
train_dataset=FoodDataset(tran_images,train_labels,transform=transform)
val_dataset=FoodDataset(val_images,val_labels,transform=transform)


In [58]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2) #num_workers is like thread |parallel loading for speed
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=2)

In [59]:
# pretrained model
model= mobilenet_v2(weights="IMAGENET1K_V1") #pretained_model | lightweight | CNN | 1000 class | million
model.classifier[1]=nn.Linear(model.classifier[1].in_features,NUM_CLASSES) # fine_tuning | backbone | model layer freeze


In [60]:
device = torch.device('cuda' if torch.cuda.is_available()else "cpu")
print("device:",device)
model=model.to(device)

device: cuda


In [61]:
criterion = nn.CrossEntropyLoss() # Loss Function | 70% burger | 30% pilaf
optimizer = optim.Adam(model.parameters(),lr=0.001) # weight
torch.backends.cudnn.benchmark = True # Benchmark Setting | Trick | 10-20% more speed


In [62]:
import torch

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):

    # TRAINING
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    # VALIDATION
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total

    avg_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Loss: {avg_loss:.4f}, "
        f"Val Accuracy: {val_acc:.2f}%"
    )

    if val_acc > best_accuracy:

        best_accuracy = val_acc

        torch.save(
            model.state_dict(),
            "/content/menu_detector.pth"
        )

        print("Saved new best model!")

Epoch [1/10] Loss: 0.5320, Val Accuracy: 88.95%
Saved new best model!
Epoch [2/10] Loss: 0.2959, Val Accuracy: 88.72%
Epoch [3/10] Loss: 0.2064, Val Accuracy: 91.28%
Saved new best model!
Epoch [4/10] Loss: 0.1870, Val Accuracy: 88.60%
Epoch [5/10] Loss: 0.1618, Val Accuracy: 87.91%
Epoch [6/10] Loss: 0.1710, Val Accuracy: 90.23%
Epoch [7/10] Loss: 0.1358, Val Accuracy: 88.72%
Epoch [8/10] Loss: 0.1185, Val Accuracy: 86.40%
Epoch [9/10] Loss: 0.1125, Val Accuracy: 89.88%
Epoch [10/10] Loss: 0.1128, Val Accuracy: 86.98%
